In [1]:
%cd ..

/Users/niccolotogni/Repo/prompt-forge


In [4]:
import logging
import os

from openai import AzureOpenAI
from dotenv import load_dotenv

from prompt_forge import (
    Project,
    LLMMessage,
    LLMResponse,
)

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)


# ── Azure OpenAI client ───────────────────────────────────────────────────────
class AzureClient:
    ENDPOINT = "https://sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com/"
    API_VERSION = "2024-12-01-preview"
    DEPLOYMENT = "gpt-5.3-chat"  # ← change to your deployment name

    def __init__(self):
        self.client = AzureOpenAI(
            api_version=self.API_VERSION,
            azure_endpoint=self.ENDPOINT,
            api_key=os.environ["AZURE_OPENAI_API_KEY"],
        )

    def complete(self, messages: list[LLMMessage], **kwargs) -> LLMResponse:
        # Strip kwargs that AzureOpenAI doesn't accept (e.g. custom keys)
        allowed = {"temperature", "max_tokens", "top_p", "frequency_penalty", "presence_penalty"}
        oai_kwargs = {k: v for k, v in kwargs.items() if k in allowed}

        resp = self.client.chat.completions.create(
            model=self.DEPLOYMENT,
            messages=[{"role": m.role, "content": m.content} for m in messages],
            **oai_kwargs,
        )
        return LLMResponse(
            text=resp.choices[0].message.content,
            usage={
                "input_tokens": resp.usage.prompt_tokens,
                "output_tokens": resp.usage.completion_tokens,
            },
        )


In [5]:
# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR = "examples/extraction_example"  # Directory containing example subfolders (001/, 002/, ...)
PROJECT_DIR = ".projects/extraction_example"       # Directory to store project files (prompt versions, logs, etc.)

BUNDLE_SCHEMA = {
    "input": ".txt",
    "expected_output": ".json",
}

SEED_PROMPT = (
    "You are a helpful assistant. Process the input and produce the expected output."
)

CONTEXT = (
    "Debug task: echo or transform text inputs."
)

# Set to "none" to skip evaluation (faster iteration), or use
# "similarity", "exact_match", "json_fields", "llm_judge"
EVAL_STRATEGY = "json_fields"


In [6]:
load_dotenv()
llm = AzureClient()
project = Project(
    name="debug_project",
    llm=llm,
    project_dir=PROJECT_DIR,
)

project.set_bundle_schema(**BUNDLE_SCHEMA)
project.set_context(CONTEXT)
project.set_seed_prompt(SEED_PROMPT)

n = project.add_examples_from_directory(DATA_DIR)
print(f"\nLoaded {n} examples")

if n == 0:
    print(
        "\nNo examples found. Create them under:\n"
        f"  {DATA_DIR}/001/input.txt\n"
        f"  {DATA_DIR}/001/expected_output.txt\n"
        "Then re-run."
    )


2026-03-11 18:15:07,178 [INFO] prompt_forge.project: Loaded project config: debug_project
2026-03-11 18:15:07,179 [INFO] prompt_forge.project: Bundle schema set: {'input': '.txt', 'expected_output': '.json'}
2026-03-11 18:15:07,180 [INFO] prompt_forge.project: Seed prompt set.
2026-03-11 18:15:07,180 [INFO] prompt_forge.bundle: Loading bundles from subdirectories in examples/extraction_example...
2026-03-11 18:15:07,182 [INFO] prompt_forge.project: Loaded 20 example bundles from examples/extraction_example



Loaded 20 examples


In [8]:
from prompt_forge.training.pipeline import TrainingConfig
from prompt_forge.utils import train_val_split

SEED = 42

train_bundles, val_bundles = train_val_split(project.bundles, val_ratio=0.2, seed=SEED)
train_config = TrainingConfig(
    batch_size=20,
    max_iterations=1,
    inference_temperature=1,
    val_max_tokens=5000,
    seed=SEED
)
print(f"\nStarting training on {project.num_examples} examples...")

report = project.train(
    config=train_config,
    eval_strategy=EVAL_STRATEGY,
    train_bundles=train_bundles,
    val_bundles=val_bundles
)

print("\n── Training Report ─────────────────────────────────────────")
print(f"Final version : v{report.final_version}")
print(f"Final score   : {report.final_score}")
print(f"Total tokens  : {report.total_tokens_used:,}")
print(f"Refinement rec: {report.refinement_recommended}")
print()


2026-03-11 18:16:02,198 [INFO] prompt_forge.training.pipeline: Restored training state: 6 previous iterations, 319851 tokens used so far
2026-03-11 18:16:02,199 [INFO] prompt_forge.training.pipeline: Starting training: 16 train examples, val_size=4, batch_size=20, max_iterations=1
2026-03-11 18:16:02,199 [INFO] prompt_forge.training.pipeline: 
Iteration 1/1
2026-03-11 18:16:02,199 [INFO] prompt_forge.training.pipeline: Selected batch: ['spec_05', 'spec_18', 'spec_13', 'spec_20', 'spec_19', 'spec_06', 'spec_02', 'spec_03', 'spec_15', 'spec_10', 'spec_14', 'spec_17', 'spec_12', 'spec_16', 'spec_11', 'spec_07']
2026-03-11 18:16:02,202 [INFO] prompt_forge.inference.agent: Batch inference: 4 inputs → 1 chunk(s)
2026-03-11 18:16:02,340 [DEBUG] httpcore.connection: connect_tcp.started host='sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com' port=443 local_address=None timeout=5.0 socket_options=None



Starting training on 20 examples...


2026-03-11 18:16:02,492 [DEBUG] httpcore.connection: connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x1119e78c0>
2026-03-11 18:16:02,492 [DEBUG] httpcore.connection: start_tls.started ssl_context=<ssl.SSLContext object at 0x110d543b0> server_hostname='sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com' timeout=5.0
2026-03-11 18:16:02,719 [DEBUG] httpcore.connection: start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x11195d810>
2026-03-11 18:16:02,719 [DEBUG] httpcore.http11: send_request_headers.started request=<Request [b'POST']>
2026-03-11 18:16:02,720 [DEBUG] httpcore.http11: send_request_headers.complete
2026-03-11 18:16:02,720 [DEBUG] httpcore.http11: send_request_body.started request=<Request [b'POST']>
2026-03-11 18:16:02,721 [DEBUG] httpcore.http11: send_request_body.complete
2026-03-11 18:16:02,721 [DEBUG] httpcore.http11: receive_response_headers.started request=<Request [b'POST']>
2026-03-11 18:16:24,985 [DEBUG] h


── Training Report ─────────────────────────────────────────
Final version : v6
Final score   : 0.95
Total tokens  : 30,007
Refinement rec: False



In [9]:
for r in report:
    status = "✓" if r.improved else "✗"
    score_str = (
        f"{r.score_before:.3f} → {r.score_after:.3f}"
        if r.score_before is not None and r.score_after is not None
        else "no eval"
    )
    tokens_str = f"  [{r.tokens_used:,} tok]" if r.tokens_used else ""
    print(f"  Iter {r.iteration}: {score_str} {status}{tokens_str}")
    print(f"    Learned: {r.learnings[:120]}")

print("\n── Prompt versions ─────────────────────────────────────────")
for v in project.list_versions():
    score = f"score={v.eval_score:.3f}" if v.eval_score is not None else "no score"
    print(f"  v{v.version}  {score}  {(v.training_log_entry or '')[:80]}")

print("\n── Inference test ──────────────────────────────────────────")
agent = project.get_inference_agent()
print(agent.prompt_info)
result = agent.run(input_text="Hello, this is a test input.")
print(f"Output: {result}")


2026-03-11 18:22:00,015 [INFO] prompt_forge.inference.agent: Loaded prompt version 6 (score: 0.98)
2026-03-11 18:22:00,017 [DEBUG] httpcore.connection: close.started
2026-03-11 18:22:00,017 [DEBUG] httpcore.connection: close.complete
2026-03-11 18:22:00,018 [DEBUG] httpcore.connection: connect_tcp.started host='sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com' port=443 local_address=None timeout=5.0 socket_options=None
2026-03-11 18:22:00,205 [DEBUG] httpcore.connection: connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x110d0ec10>
2026-03-11 18:22:00,206 [DEBUG] httpcore.connection: start_tls.started ssl_context=<ssl.SSLContext object at 0x110d543b0> server_hostname='sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com' timeout=5.0


  Iter 1: 0.970 → 0.950 ✗  [24,093 tok]
    Learned: - Some brands appearing in all caps should be normalized to stylized casing (e.g., LOGIMOVE → LogiMove, PYROMAX → PyroMa

── Prompt versions ─────────────────────────────────────────
  v0  no score  Seed prompt
  v1  score=0.922  - Product names must exclude descriptive machine-type suffixes (e.g., remove "Pr
  v2  score=0.978  - Unit conversions must be rounded consistently: dimensions and weight to whole 
  v3  score=0.982  - Throughput units must also normalize “per shift” → “/shift” and remove thousan
  v4  score=0.976  - Ambient temperature limits must be preferred over internal/process temperature
  v5  no score  - Manufacturer may appear as “Distributed by” or other supplier indicators and s
  v6  score=0.980  - Throughput may originate from phrases like “cycle throughput”, “flow rate”, or

── Inference test ──────────────────────────────────────────
Prompt: 491 lines, 13547 chars, output_schema=['product_name', 'manufacturer'

2026-03-11 18:22:00,422 [DEBUG] httpcore.connection: start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x111ab8770>
2026-03-11 18:22:00,423 [DEBUG] httpcore.http11: send_request_headers.started request=<Request [b'POST']>
2026-03-11 18:22:00,423 [DEBUG] httpcore.http11: send_request_headers.complete
2026-03-11 18:22:00,423 [DEBUG] httpcore.http11: send_request_body.started request=<Request [b'POST']>
2026-03-11 18:22:00,424 [DEBUG] httpcore.http11: send_request_body.complete
2026-03-11 18:22:00,424 [DEBUG] httpcore.http11: receive_response_headers.started request=<Request [b'POST']>
2026-03-11 18:22:05,690 [DEBUG] httpcore.http11: receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Length', b'2072'), (b'Content-Type', b'application/json'), (b'apim-request-id', b'f7b767fc-bdbb-4dc5-b6c0-731762fe27a5'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'x-content-type-options', b'nosniff'), (b'x

Output: {'product_name': None, 'manufacturer': None, 'part_number': None, 'year_of_manufacture': None, 'machine_type': None, 'application_sector': None, 'power_supply': {'voltage_v': None, 'phases': None, 'frequency_hz': None, 'power_rating_kw': None}, 'dimensions': {'length_mm': None, 'width_mm': None, 'height_mm': None}, 'weight_kg': None, 'operating_conditions': {'temperature_min_c': None, 'temperature_max_c': None, 'humidity_max_pct': None}, 'performance': {'max_speed_rpm': None, 'throughput': None, 'accuracy_mm': None, 'noise_level_db': None}, 'safety_certifications': [], 'communication_interfaces': [], 'maintenance_interval_hours': None, 'warranty_years': None}


In [10]:
print(project.list_versions()[2].prompt_text)

You are an information extraction system that converts industrial equipment specification text into a structured JSON record.

Your task: read the provided technical description and extract machine specifications into a STRICT JSON object that exactly follows the schema and rules below.

CRITICAL OUTPUT RULES
- Output MUST be valid JSON.
- Output ONLY the JSON object. No explanations, no comments, no extra text.
- Use exactly the field names defined in the schema.
- Include all fields even if the value is unknown.
- If a value is missing or cannot be determined, use null (not "null", not empty string).
- Arrays must be valid JSON arrays.
- Numbers must be numeric types, not strings.
- All numeric values must be rounded appropriately after unit conversions (rules below).

ROUNDING AND NUMERIC NORMALIZATION (CRITICAL)
When conversions produce decimal values:
- Dimensions in millimeters must be rounded to the nearest whole millimeter (integer).
  Example: 2387.6 → 2388, 1066.8 → 1067.
- W

In [11]:
print(report.iterations[0].learnings)

- Some brands appearing in all caps should be normalized to stylized casing (e.g., LOGIMOVE → LogiMove, PYROMAX → PyroMax, CHILLPRO → ChillPro).
- Safety certifications often include descriptors like “Listed” or “certified” that must be removed while keeping the core standard.
- Battery-powered machines should still use mains charging voltage for the supply field but may use drive/motor power as the power rating.
- Throughput must return null when the text explicitly states that production rate varies with conditions.


In [12]:
print(report.iterations[0].issues)

- Brand capitalization normalization still relies on heuristic interpretation; a definitive brand dictionary would improve consistency.
- Some borderline cases remain for dimension derivation (working envelope vs physical size) when both appear but neither is clearly labeled as overall dimensions.
